In [1]:
# ============================================================
# Smart Container Risk Detection Engine
# Hybrid Ensemble: XGBoost + Random Forest + Isolation Forest
# ============================================================

# --- 1. Install Dependencies ---
# !pip install pandas numpy scikit-learn xgboost openpyxl -q

In [2]:
# --- 2. Import Libraries ---

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
)
from xgboost import XGBClassifier

In [3]:
# --- 3. Load Datasets ---

COL_RENAME = {
    "Declaration_Date (YYYY-MM-DD)": "Declaration_Date",
    "Trade_Regime (Import / Export / Transit)": "Trade_Regime",
}


def load_data(path: str) -> pd.DataFrame:
    """Load CSV and standardize column names."""
    return pd.read_csv(path).rename(columns=COL_RENAME)


df_hist = load_data("Historical Data.csv")
df_rt = load_data("Real-Time Data.csv")

In [4]:
# --- 4. Preprocessing & Label Creation ---

RISK_STATUSES = {"critical", "hold", "inspect", "detained", "flagged"}
NUM_COLS = ["Declared_Value", "Declared_Weight", "Measured_Weight", "Dwell_Time_Hours"]
CAT_COLS = [
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
]


def preprocess(df: pd.DataFrame, has_label: bool = True) -> pd.DataFrame:
    """Clean types, impute missing values, and derive binary risk label."""
    df = df.copy()

    # Temporal features
    df["Declaration_Date"] = pd.to_datetime(df["Declaration_Date"], errors="coerce")
    df["declaration_month"] = df["Declaration_Date"].dt.month.fillna(1).astype(int)
    df["declaration_dow"] = df["Declaration_Date"].dt.dayofweek.fillna(0).astype(int)

    # Numeric cleaning & imputation
    for col in NUM_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col].fillna(df[col].median(), inplace=True)
    for col in ["Declared_Value", "Declared_Weight"]:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))

    # Categorical imputation
    for col in CAT_COLS:
        df[col].fillna(df[col].mode()[0], inplace=True)

    # Binary risk label from Clearance_Status
    if has_label and "Clearance_Status" in df.columns:
        df["Risk_Label"] = (
            df["Clearance_Status"].str.strip().str.lower()
            .isin(RISK_STATUSES).astype(int)
        )

    return df


df_hist = preprocess(df_hist, has_label=True)
df_rt = preprocess(df_rt, has_label=False)

In [5]:
# --- 5. Feature Engineering (Leak-Free, Anti-Overfit) ---
#
# KEY DESIGN DECISIONS:
#   - REMOVED exporter_weight_deviation: Exporter IDs have <10% overlap
#     between train/val risky cases, so exporter-specific features memorize.
#   - All features are purely domain-driven anomaly signals that generalize.
#   - Commodity benchmarks use GLOBAL fallbacks with high min-count threshold.

EPS = 1e-5


def compute_train_stats(df: pd.DataFrame) -> dict:
    """Compute benchmark statistics from training data only."""
    stats = {}

    # Commodity (HS_Code) weight benchmarks — require >= 10 observations
    hs_grp = df.groupby("HS_Code")["Declared_Weight"]
    hs_counts = hs_grp.count()
    reliable_hs = hs_counts[hs_counts >= 10].index
    stats["hs_avg"] = hs_grp.mean().loc[reliable_hs]
    stats["hs_std"] = hs_grp.std().fillna(1).loc[reliable_hs]
    stats["hs_global_avg"] = df["Declared_Weight"].mean()
    stats["hs_global_std"] = max(df["Declared_Weight"].std(), 1)

    # Dwell time benchmarks
    stats["dwell_mean"] = df["Dwell_Time_Hours"].mean()
    stats["dwell_std"] = max(df["Dwell_Time_Hours"].std(), 1)

    # Value benchmarks per HS code
    val_grp = df.groupby("HS_Code")["Declared_Value"]
    val_counts = val_grp.count()
    reliable_val = val_counts[val_counts >= 10].index
    stats["hs_val_avg"] = val_grp.mean().loc[reliable_val]
    stats["hs_val_std"] = val_grp.std().fillna(1).loc[reliable_val]
    stats["hs_val_global_avg"] = df["Declared_Value"].mean()
    stats["hs_val_global_std"] = max(df["Declared_Value"].std(), 1)

    # Global weight stats (for weight_diff normalization)
    stats["weight_diff_mean"] = (df["Measured_Weight"] - df["Declared_Weight"]).mean()
    stats["weight_diff_std"] = max((df["Measured_Weight"] - df["Declared_Weight"]).std(), 1)

    # Global density stats
    density = df["Measured_Weight"] / (df["Declared_Value"] + EPS)
    stats["density_mean"] = density.mean()
    stats["density_std"] = max(density.std(), 1)

    # Global value_per_weight stats
    vpw = df["Declared_Value"] / (df["Declared_Weight"] + EPS)
    stats["vpw_mean"] = vpw.mean()
    stats["vpw_std"] = max(vpw.std(), 1)

    return stats


def engineer_features(df: pd.DataFrame, train_stats: dict) -> pd.DataFrame:
    """Build domain-specific features using pre-computed training statistics.
    
    All features are normalized z-scores or ratios that generalize across
    train/val splits. No entity-specific features (exporter/importer).
    """
    df = df.copy()

    # --- Weight difference features ---
    df["weight_diff"] = df["Measured_Weight"] - df["Declared_Weight"]
    df["weight_ratio"] = df["Measured_Weight"] / (df["Declared_Weight"] + EPS)
    df["weight_deviation_pct"] = (
        df["weight_diff"].abs() / (df["Declared_Weight"] + EPS)
    ).clip(upper=0.5)

    # Normalized weight diff z-score (generalizes better than raw diff)
    df["weight_diff_zscore"] = (
        (df["weight_diff"] - train_stats["weight_diff_mean"])
        / (train_stats["weight_diff_std"] + EPS)
    ).clip(-3, 3)

    # --- Commodity weight benchmark ---
    commodity_avg = df["HS_Code"].map(train_stats["hs_avg"]).fillna(train_stats["hs_global_avg"])
    commodity_std = df["HS_Code"].map(train_stats["hs_std"]).fillna(train_stats["hs_global_std"])
    df["commodity_weight_zscore"] = (
        (df["Declared_Weight"] - commodity_avg) / (commodity_std + EPS)
    ).clip(-3, 3)

    # --- Commodity value benchmark ---
    val_avg = df["HS_Code"].map(train_stats["hs_val_avg"]).fillna(train_stats["hs_val_global_avg"])
    val_std = df["HS_Code"].map(train_stats["hs_val_std"]).fillna(train_stats["hs_val_global_std"])
    df["commodity_value_zscore"] = (
        (df["Declared_Value"] - val_avg) / (val_std + EPS)
    ).clip(-3, 3)

    # --- Cargo density (normalized) ---
    df["density"] = df["Measured_Weight"] / (df["Declared_Value"] + EPS)
    df["density_zscore"] = (
        (df["density"] - train_stats["density_mean"])
        / (train_stats["density_std"] + EPS)
    ).clip(-3, 3)

    # --- Value per weight (normalized) ---
    df["value_per_weight"] = df["Declared_Value"] / (df["Declared_Weight"] + EPS)
    df["vpw_zscore"] = (
        (df["value_per_weight"] - train_stats["vpw_mean"])
        / (train_stats["vpw_std"] + EPS)
    ).clip(-3, 3)

    # --- Dwell time z-score ---
    df["dwell_time_zscore"] = (
        (df["Dwell_Time_Hours"] - train_stats["dwell_mean"])
        / (train_stats["dwell_std"] + EPS)
    ).clip(-3, 3)

    return df

In [6]:
# --- 6. Encode, Split FIRST, Then Engineer (Leak-Free Pipeline) ---
#
# The key anti-overfit strategy:
#   1. Split raw data into train / val BEFORE any statistics
#   2. Compute all benchmark stats from TRAIN split only
#   3. Apply those stats to both train and val
#   4. NO entity-specific features (exporter/importer ID removed)

FEATURE_COLS = [
    # Categorical
    "Trade_Regime", "Origin_Country", "Destination_Country",
    "Destination_Port", "Shipping_Line",
    # Numeric base
    "Declared_Value", "Declared_Weight", "Measured_Weight",
    "Dwell_Time_Hours", "declaration_month", "declaration_dow",
    # Weight anomaly features (domain-driven, generalizable)
    "weight_diff", "weight_ratio", "weight_deviation_pct", "weight_diff_zscore",
    # Commodity benchmarks
    "commodity_weight_zscore",
    "commodity_value_zscore",
    # Cargo density & value (normalized)
    "density_zscore", "vpw_zscore",
    # Dwell time
    "dwell_time_zscore",
]


def encode_categories(df, fit=True, encoders=None):
    """Label-encode categorical columns."""
    df = df.copy()
    if encoders is None:
        encoders = {}
    for col in CAT_COLS:
        if col not in df.columns:
            continue
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
        else:
            le = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].astype(str).apply(
                lambda v: v if v in known else le.classes_[0]
            )
            df[col] = le.transform(df[col])
    return df, encoders


def prepare_features(df):
    """Select feature columns and target."""
    avail = [c for c in FEATURE_COLS if c in df.columns]
    X = df[avail]
    y = df["Risk_Label"].astype(int) if "Risk_Label" in df.columns else None
    return X, y


# Step 1: Stratified split on RAW preprocessed data (before any feature engineering)
indices = df_hist.index.values
y_raw = df_hist["Risk_Label"].values
tr_idx, val_idx = train_test_split(
    indices, test_size=0.2, stratify=y_raw, random_state=42
)
df_train_raw = df_hist.loc[tr_idx].copy()
df_val_raw = df_hist.loc[val_idx].copy()

# Step 2: Compute stats from TRAINING split only
train_stats = compute_train_stats(df_train_raw)

# Step 3: Engineer features using train-only stats
df_train_eng = engineer_features(df_train_raw, train_stats)
df_val_eng = engineer_features(df_val_raw, train_stats)

# Step 4: Fit encoders on training split only
df_train_enc, encoders = encode_categories(df_train_eng, fit=True)
df_val_enc, _ = encode_categories(df_val_eng, fit=False, encoders=encoders)

# Step 5: Extract feature matrices
X_train, y_train = prepare_features(df_train_enc)
X_val, y_val = prepare_features(df_val_enc)

# Step 6: Standardize numeric features
scaler = StandardScaler()
num_feats = [c for c in X_train.columns if c not in CAT_COLS]
X_train[num_feats] = scaler.fit_transform(X_train[num_feats])
X_val[num_feats] = scaler.transform(X_val[num_feats])

print(f"Training set:   {X_train.shape[0]:,} rows, {(y_train==1).sum()} risky ({(y_train==1).mean()*100:.2f}%)")
print(f"Validation set: {X_val.shape[0]:,} rows, {(y_val==1).sum()} risky ({(y_val==1).mean()*100:.2f}%)")
print(f"Features used:  {X_train.shape[1]}")
print(f"Feature list:   {list(X_train.columns)}")

Training set:   43,200 rows, 436 risky (1.01%)
Validation set: 10,800 rows, 109 risky (1.01%)
Features used:  20
Feature list:   ['Trade_Regime', 'Origin_Country', 'Destination_Country', 'Destination_Port', 'Shipping_Line', 'Declared_Value', 'Declared_Weight', 'Measured_Weight', 'Dwell_Time_Hours', 'declaration_month', 'declaration_dow', 'weight_diff', 'weight_ratio', 'weight_deviation_pct', 'weight_diff_zscore', 'commodity_weight_zscore', 'commodity_value_zscore', 'density_zscore', 'vpw_zscore', 'dwell_time_zscore']


In [8]:
# --- 8. Ensemble Prediction & Risk Scoring ---
#
# Ensemble weights: XGBoost (calibrated) gets most weight since it's
# the strongest supervised learner. IF provides unsupervised signal.

W_XGB, W_RF, W_IF = 0.50, 0.30, 0.20


def normalize_isolation_scores(model, X_if, baseline_raw=None):
    """Convert Isolation Forest decision function to 0-1 anomaly probability."""
    raw = -model.decision_function(X_if)
    if baseline_raw is not None:
        lo, hi = np.percentile(baseline_raw, 1), np.percentile(baseline_raw, 99)
    else:
        lo, hi = np.percentile(raw, 1), np.percentile(raw, 99)
    return np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1)


def compute_ensemble_score(xgb, rf, iso, X, X_if, baseline_raw=None):
    """Weighted ensemble of all three models, normalized to 0-100."""
    xgb_prob = xgb.predict_proba(X)[:, 1]
    rf_prob = rf.predict_proba(X)[:, 1]
    if_score = normalize_isolation_scores(iso, X_if, baseline_raw)
    combined = W_XGB * xgb_prob + W_RF * rf_prob + W_IF * if_score
    return np.round(combined * 100, 2)


def assign_risk_label(score, threshold):
    """Convert numeric risk score to categorical label using data-driven threshold."""
    high_threshold = threshold + (100 - threshold) * 0.5
    if score >= high_threshold:
        return "High Risk"
    if score >= threshold:
        return "Medium Risk"
    return "Low Risk"


# Score train & validation sets
train_scores = compute_ensemble_score(
    xgb_model, rf_model, iso_model, X_train, X_train[IF_FEATURES], train_if_raw
)
val_scores = compute_ensemble_score(
    xgb_model, rf_model, iso_model, X_val, X_val[IF_FEATURES], train_if_raw
)

print(f"Train scores — min: {train_scores.min():.2f}, median: {np.median(train_scores):.2f}, max: {train_scores.max():.2f}")
print(f"Val scores   — min: {val_scores.min():.2f}, median: {np.median(val_scores):.2f}, max: {val_scores.max():.2f}")
print(f"\nTrain risky scores:  mean={train_scores[y_train==1].mean():.2f}, std={train_scores[y_train==1].std():.2f}")
print(f"Val risky scores:    mean={val_scores[y_val==1].mean():.2f}, std={val_scores[y_val==1].std():.2f}")
print(f"Score gap (risky):   {train_scores[y_train==1].mean() - val_scores[y_val==1].mean():.2f}")

Train scores — min: 0.82, median: 2.98, max: 99.19
Val scores   — min: 0.85, median: 3.02, max: 99.13

Train risky scores:  mean=69.35, std=24.87
Val risky scores:    mean=67.21, std=26.81
Score gap (risky):   2.14


In [9]:
# --- 8b. Model Evaluation — Data-Driven Threshold + Overfit Check ---
#
# We find the optimal threshold from the VALIDATION set's Precision-Recall
# curve (maximizing F1). This gives us the best trade-off for the minority class.

# --- Step 1: Find optimal threshold from PR curve on VALIDATION set ---
val_probs = val_scores / 100.0
precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_val, val_probs)

# F1 at each threshold
f1_arr = 2 * (precision_arr[:-1] * recall_arr[:-1]) / (precision_arr[:-1] + recall_arr[:-1] + 1e-9)
best_idx = np.argmax(f1_arr)
THRESHOLD = np.round(thresholds_arr[best_idx] * 100, 2)  # convert back to 0-100 scale

print(f"{'=' * 55}")
print(f"  Optimal Threshold (from PR curve): {THRESHOLD:.2f}")
print(f"  At this threshold — Precision: {precision_arr[best_idx]:.4f}, Recall: {recall_arr[best_idx]:.4f}")
print(f"{'=' * 55}")


# --- Step 2: Evaluate both sets at optimal threshold ---
def evaluate_set(tag, y_true, scores, threshold):
    """Compute and display confusion matrix + all classification metrics."""
    y_pred = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    auc = roc_auc_score(y_true, scores / 100)
    ap = average_precision_score(y_true, scores / 100)
    print(f"\n{'=' * 55}")
    print(f"  {tag} Set Evaluation  (threshold={threshold:.2f})")
    print(f"{'=' * 55}")
    print(f"  Confusion Matrix:")
    print(f"    TN={cm[0][0]:>6}   FP={cm[0][1]:>6}")
    print(f"    FN={cm[1][0]:>6}   TP={cm[1][1]:>6}")
    print(f"\n  Accuracy:          {acc:.4f}")
    print(f"  Precision:         {prec:.4f}")
    print(f"  Recall:            {rec:.4f}")
    print(f"  F1-Score:          {f1:.4f}")
    print(f"  F1-Score (wt):     {f1_w:.4f}")
    print(f"  AUC-ROC:           {auc:.4f}")
    print(f"  Avg Precision (AP):{ap:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred,
          target_names=["Safe", "Risky"], zero_division=0))
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "f1_weighted": f1_w, "auc_roc": auc, "avg_precision": ap}


train_metrics = evaluate_set("Train", y_train, train_scores, THRESHOLD)
val_metrics = evaluate_set("Validation", y_val, val_scores, THRESHOLD)

# --- Step 3: Overfit gap analysis ---
print(f"\n{'=' * 55}")
print(f"  Overfit Gap (Train - Validation)")
print(f"{'=' * 55}")
for key in train_metrics:
    gap = train_metrics[key] - val_metrics[key]
    flag = " ⚠️" if abs(gap) > 0.10 else " ✓"
    print(f"  {key:>15s}: {gap:+.4f}{flag}")

# --- Step 4: Cross-validation (on XGBoost base before calibration) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(xgb_base, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"\n  5-Fold CV AUC (XGB base): {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

# --- Step 5: Risk label distribution ---
val_labels = pd.Series([assign_risk_label(s, THRESHOLD) for s in val_scores])
print(f"\n  Validation Risk Distribution: {val_labels.value_counts().to_dict()}")

  Optimal Threshold (from PR curve): 58.11
  At this threshold — Precision: 0.9870, Recall: 0.6972

  Train Set Evaluation  (threshold=58.11)
  Confusion Matrix:
    TN= 42754   FP=    10
    FN=   119   TP=   317

  Accuracy:          0.9970
  Precision:         0.9694
  Recall:            0.7271
  F1-Score:          0.8309
  F1-Score (wt):     0.9968
  AUC-ROC:           0.9917
  Avg Precision (AP):0.8519

  Classification Report:
              precision    recall  f1-score   support

        Safe       1.00      1.00      1.00     42764
       Risky       0.97      0.73      0.83       436

    accuracy                           1.00     43200
   macro avg       0.98      0.86      0.91     43200
weighted avg       1.00      1.00      1.00     43200


  Validation Set Evaluation  (threshold=58.11)
  Confusion Matrix:
    TN= 10690   FP=     1
    FN=    33   TP=    76

  Accuracy:          0.9969
  Precision:         0.9870
  Recall:            0.6972
  F1-Score:          0.8172
  F

In [10]:
# --- 9. Explanation Generation ---

FEAT_DESC = {
    "weight_diff":              "High weight deviation between declared and measured",
    "weight_ratio":             "Abnormal measured-to-declared weight ratio",
    "weight_deviation_pct":     "Significant weight deviation detected",
    "weight_diff_zscore":       "Abnormal weight discrepancy for the dataset",
    "commodity_weight_zscore":  "Weight unusual for this commodity type",
    "commodity_value_zscore":   "Declared value unusual for this commodity type",
    "density_zscore":           "Abnormal cargo density detected",
    "vpw_zscore":               "Unusual value-to-weight ratio",
    "dwell_time_zscore":        "Abnormal dwell time in port",
    "Dwell_Time_Hours":         "Extended port dwell time",
    "Declared_Value":           "Unusual declared cargo value",
    "Declared_Weight":          "Unusual declared weight",
    "Measured_Weight":          "Unusual measured weight",
}


def generate_explanations(model, X: pd.DataFrame, scores: np.ndarray, threshold: float) -> list:
    """Feature-importance-based human-readable explanations.
    
    Uses the Random Forest's feature_importances_ since the calibrated
    XGBoost doesn't directly expose them.
    """
    importances = model.feature_importances_
    feat_names = X.columns.tolist()
    top_indices = np.argsort(importances)[::-1][:6]
    top_feats = [feat_names[i] for i in top_indices]

    explanations = []
    for idx in range(len(X)):
        score = scores[idx]
        if score < threshold:
            explanations.append("No significant risk signals detected.")
            continue

        row = X.iloc[idx]
        signals = []
        for feat in top_feats:
            desc = FEAT_DESC.get(feat, feat.replace("_", " ").title())
            val = row[feat]
            if feat in ("commodity_weight_zscore", "commodity_value_zscore",
                        "dwell_time_zscore", "weight_diff_zscore",
                        "density_zscore", "vpw_zscore"):
                if abs(val) > 1.0:
                    signals.append(desc)
            elif feat == "weight_deviation_pct":
                if val > 0.08:
                    signals.append(desc)
            else:
                signals.append(desc)
            if len(signals) >= 3:
                break

        high_threshold = threshold + (100 - threshold) * 0.5
        if not signals:
            explanations.append(
                "Ensemble model detected elevated risk pattern."
                if score >= high_threshold
                else "Moderate risk signals detected by ensemble model."
            )
        else:
            explanations.append("; ".join(signals) + ".")

    return explanations

In [11]:
# --- 10. Real-Time Inference & Output DataFrame ---


def run_inference(df_new: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Full inference pipeline: engineer -> encode -> scale -> score -> explain."""
    df_eng = engineer_features(df_new, train_stats)
    df_enc, _ = encode_categories(df_eng, fit=False, encoders=encoders)
    X_new, _ = prepare_features(df_enc)
    X_new[num_feats] = scaler.transform(X_new[num_feats])

    scores = compute_ensemble_score(
        xgb_model, rf_model, iso_model,
        X_new, X_new[IF_FEATURES], train_if_raw
    )
    return pd.DataFrame({
        "container_id": df_new["Container_ID"].values,
        "risk_score": scores,
        "risk_label": [assign_risk_label(s, threshold) for s in scores],
        "explanation": generate_explanations(rf_model, X_new, scores, threshold),
    })


# Historical predictions
results_hist = run_inference(df_hist, THRESHOLD)
results_hist.to_csv("risk_predictions.csv", index=False)

# Real-time predictions
results_rt = run_inference(df_rt, THRESHOLD)
results_rt.to_csv("risk_predictions_realtime.csv", index=False)

print(f"Threshold used: {THRESHOLD}")
print(f"\nHistorical risk distribution:")
print(results_hist["risk_label"].value_counts())
print(f"\nReal-time risk distribution:")
print(results_rt["risk_label"].value_counts())
results_rt.head(10)

Threshold used: 58.11

Historical risk distribution:
risk_label
Low Risk       53596
High Risk        245
Medium Risk      159
Name: count, dtype: int64

Real-time risk distribution:
risk_label
Low Risk       8424
High Risk        30
Medium Risk      27
Name: count, dtype: int64


,container_id,risk_score,risk_label,explanation
0,41256141,1.61,Low Risk,No significant risk signals detected.
1,20889431,4.30,Low Risk,No significant risk signals detected.
2,25403940,5.24,Low Risk,No significant risk signals detected.
3,43489778,7.86,Low Risk,No significant risk signals detected.
4,94895240,2.71,Low Risk,No significant risk signals detected.
5,81282547,30.21,Low Risk,No significant risk signals detected.
6,13935563,6.48,Low Risk,No significant risk signals detected.
7,57224212,8.19,Low Risk,No significant risk signals detected.
8,21579017,18.76,Low Risk,No significant risk signals detected.
9,97545517,2.44,Low Risk,No significant risk signals detected.
